# Build a File Upload Chatbot, Step by Step

Extends Level 1 by adding the ability to send files alongside messages. The AI can read documents, analyze images, and process data files.

**What you'll learn:**
- **Base64 encoding** — converting binary files to text for the API
- **MIME types** — telling the API what kind of file you're sending
- **Multimodal input** — mixing text and files in one message
- **Different file type handling** (images vs PDFs vs text files)

**Prerequisites:** Complete the Level 1 notebook first, or have your `.env` configured.

## Setup

Same as Level 1, plus two new standard library imports: `base64` (converts binary files to text) and `mimetypes` (guesses file types from extensions).

In [ ]:
import os
import base64      # Converts binary file data to text (API-safe)
import mimetypes   # Guesses file types from extensions (.pdf -> application/pdf)
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables
env_path = Path.cwd().parent / ".env"
if not env_path.exists():
    env_path = Path.cwd() / ".env"
load_dotenv(env_path)

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
MODEL = os.getenv("MODEL_NAME", "gpt-5.2")
REASONING_EFFORT = os.getenv("REASONING_EFFORT", "low")
INSTRUCTIONS = os.getenv(
    "CHATBOT_INSTRUCTIONS",
    "You are a helpful assistant. Be concise and friendly.",
)

if not ENDPOINT or not API_KEY:
    print("ERROR: Missing configuration!")
    print(f"  Looked for .env at: {env_path}")
    print("  Make sure AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY are set.")
else:
    client = OpenAI(base_url=ENDPOINT, api_key=API_KEY, max_retries=10)
    print("Ready! Client created.")

## How File Upload Works

The API accepts different file types in different formats:

| File Type | API Format | Example |
|-----------|-----------|---------|
| Images (.png, .jpg) | `input_image` with data URI | Photos, screenshots |
| PDFs (.pdf) | `input_file` with base64 data | Documents, papers |
| Text files (.txt, .csv, .py) | `input_text` with file content inline | Code, data, notes |

The key idea: instead of sending `input="your message"` (a simple string), you build a **list of content items** that can mix files and text in one message.

## Step 1: Encoding Files

The API expects text (JSON), but files are binary. **Base64** converts binary data into ASCII text so it can be safely included in a JSON request.

We also need to tell the API what kind of file we're sending. That's what **MIME types** do — they map file extensions to content types (e.g., `.pdf` → `application/pdf`, `.png` → `image/png`).

In [ ]:
def encode_file(file_path: str) -> tuple[str, str, str]:
    """Read a file and prepare it for the API.
    Returns: (filename, mime_type, base64_data)
    """
    path = Path(file_path)
    mime_type = mimetypes.guess_type(str(path))[0] or "application/octet-stream"
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return path.name, mime_type, data

# Let's try it with the sample text file
name, mime, data = encode_file("sample-files/sample.txt")
print(f"Filename:  {name}")
print(f"MIME type: {mime}")
print(f"Base64:    {data[:80]}...")
print(f"Length:    {len(data)} characters")

## Step 2: Sending a Text File

For text files (`.txt`, `.csv`, `.py`, etc.) we can read the content as plain text and include it directly in the prompt. This is simpler than base64 and works well since the content is already text.

Notice how `input` changes from a simple string to a **list with a message object** containing multiple content items.

In [ ]:
# Read the sample text file
file_path = "sample-files/sample.txt"
file_content = Path(file_path).read_text()

print(f"File content:\n{file_content}\n")
print("---\n")

# Send it to the AI by including the text in the input array
response = client.responses.create(
    model=MODEL,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": f"[Attached file: sample.txt]\n\n{file_content}"},
            {"type": "input_text", "text": "Summarize this file in one sentence."},
        ],
    }],
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
)

print(f"Assistant: {response.output_text}")

## Step 3: Sending an Image

Images use `input_image` with a **data URI** — a string that looks like `data:image/jpeg;base64,/9j/4AAQ...`. The AI can describe, analyze, and answer questions about images.

In [ ]:
# Encode the sample image
name, mime, b64data = encode_file("sample-files/beach.jpg")
print(f"Image: {name} ({mime})")
print(f"Base64 size: {len(b64data):,} characters\n")

# Send the image with a question
response = client.responses.create(
    model=MODEL,
    input=[{
        "role": "user",
        "content": [
            {
                "type": "input_image",
                "image_url": f"data:{mime};base64,{b64data}",
            },
            {"type": "input_text", "text": "Describe this image in detail."},
        ],
    }],
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
)

print(f"Assistant: {response.output_text}")

## Step 4: The Input Array Pattern

Here's the general pattern for sending files. Instead of `input="your message"`, you build a structured list:

```python
input=[{
    "role": "user",
    "content": [
        {"type": "input_image", "image_url": "data:image/png;base64,..."},   # An image
        {"type": "input_file", "filename": "doc.pdf", "file_data": "..."},   # A PDF
        {"type": "input_text", "text": "[Attached file: data.csv]\n\n..."},  # Text file content
        {"type": "input_text", "text": "Your question here"},                # Your message
    ]
}]
```

You can mix and match — send multiple files of different types in a single message.

## Step 5: Conversation Chaining with Files

`previous_response_id` works the same way as Level 1. The API remembers the files you sent in previous turns, so you can ask follow-up questions about them without re-sending the file.

In [ ]:
# Turn 1: Send the image and ask about it
name, mime, b64data = encode_file("sample-files/beach.jpg")

response1 = client.responses.create(
    model=MODEL,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_image", "image_url": f"data:{mime};base64,{b64data}"},
            {"type": "input_text", "text": "What do you see in this image?"},
        ],
    }],
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
)

print(f"You:       [attached beach.jpg] What do you see in this image?")
print(f"Assistant: {response1.output_text}\n")

# Turn 2: Ask a follow-up — no need to re-send the image!
response2 = client.responses.create(
    model=MODEL,
    input="What emotions or mood does the image convey?",
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    previous_response_id=response1.id,
)

print(f"You:       What emotions or mood does the image convey?")
print(f"Assistant: {response2.output_text}")

## You Built a File Upload Chatbot!

You've learned every concept needed to send files to the AI:

1. **Base64 encoding** — converts binary files to API-safe text
2. **MIME types** — tell the API what kind of file you're sending
3. **Three file formats** — images (`input_image`), PDFs (`input_file`), text files (`input_text` inline)
4. **Multimodal input** — mix files and text in one message using the input array
5. **Conversation chaining** — the API remembers files from previous turns

The full terminal chatbot in `python/chat.py` adds slash commands (`/upload`, `/files`) and handles all file types automatically.

### Try it out

```bash
cd level-2-files/python
python chat.py
```

### What's next?

- **[Level 3 — Web UI](../level-3-web/)** — browser-based chat with streaming and file uploads
- **[PRDs](../PRDs/)** — workshop exercises to extend your chatbot with Claude Code